In [0]:
%pip install pendulum

In [0]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import pendulum

In [0]:
spark.sql("create catalog if not exists proyecto_final_prueba")

In [0]:
spark.sql("use catalog proyecto_final_prueba")


In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "raw"
volume = "weather"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

In [0]:
spark.sql(f"drop volume if exists {catalog}.{schema}.{volume}")

In [0]:
spark.sql(f"create volume if not exists {catalog}.{schema}.{volume}")

In [0]:

dbutils.widgets.text("start_date", "2026-08-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2026-08-15", "End Date (YYYY-MM-DD)")
dbutils.widgets.text("latitude", "-6.2294,-9.5278,-13.6339,-16.3989,-13.1588,-7.1638,-12.0566,-13.5226,-12.7826,-9.9306,-14.0678,-12.0651,-8.1159,-6.7714,-12.0432,-3.7491,-12.5933,-17.1983,-10.6675,-5.1945,-15.8402,-6.0333,-18.0146,-3.5669,-8.3791", "Latitude")
dbutils.widgets.text("longitude", "-77.8728,-77.5278,-72.8814,-71.5350,-74.2239,-78.5003,-77.1181,-71.9673,-74.9727,-76.2422,-75.7286,-75.2049,-79.0300,-79.8409,-77.0282,-73.2538,-69.1836,-70.9357,-76.2567,-80.6328,-70.0219,-76.9667,-70.2536,-80.4515,-74.5539", "Longitude")
dbutils.widgets.text("hourly_vars", "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code", "Hourly Variables")
dbutils.widgets.text("output_dir", "/Volumes/proyecto_final_prueba/raw/weather", "Output Directory")
dbutils.widgets.text("api_endpoint", "https://archive-api.open-meteo.com/v1/archive", "API endpoint")
dbutils.widgets.text("timeout", "100", "Time Out")

In [0]:
start_date = pendulum.parse(dbutils.widgets.get("start_date")).date()
end_date = pendulum.parse(dbutils.widgets.get("end_date")).date()
latitude = dbutils.widgets.get("latitude")
longitude = dbutils.widgets.get("longitude")
hourly_vars = dbutils.widgets.get("hourly_vars")
output_dir = Path(dbutils.widgets.get("output_dir"))
api_endpoint = dbutils.widgets.get("api_endpoint")
timeout = int(dbutils.widgets.get("timeout"))

In [0]:
save_files =[]

current_date = start_date
session = requests.Session()

while current_date <= end_date:
    date_str = current_date.to_date_string()

    daily_output_dir = (output_dir / f'{current_date.year}/{current_date.month}/{current_date.day}')
    daily_output_dir.mkdir(parents=True, exist_ok=True)

    file_path = daily_output_dir / f'weather_{date_str}.json'

    api_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "start_date": date_str,
        "end_date": date_str
    }

    try:
        response = session.get(api_endpoint, params=api_params, timeout=timeout)
        response.raise_for_status()
    except requests.RequestException:
        current_date = current_date.add(days=1)
        continue

    file_path.write_bytes(response.content)
    save_files.append(file_path)
    current_date = current_date.add(days=1)

print(f"Archivos guardados: {len(save_files)}")